# Analysis of the Gutsafe Regression Models: Training, Data, and Performance Metrics
*Author: Shashwath Manjunath* | *June 2026*

## Abstract
This paper presents a comprehensive analysis of the multi-output regression models developed for the **Gutsafe** platform. We evaluate two model architectures: a multi-output **Gradient Boosting Decision Tree (GBDT)** regressor and a **PyTorch Multilayer Perceptron (MLP)**. Both models predict a 7-dimensional gut microbiome perturbation vector from a 24-dimensional binary food additive presence matrix. The models were trained and validated on a dataset of $1,133$ products sourced from the Open Food Facts and USDA FoodData Central databases. 

Both models achieve extremely high precision, with test Mean Absolute Errors (MAE) of **0.00051** (GBDT) and **0.00018** (MLP), and $R^2$ scores exceeding **0.999**. We show that this performance is driven by the deterministic, noise-free, linear-additive nature of the label generation function, which the models have more than sufficient capacity to represent.

---
## 1. Introduction
The **Gutsafe** system is designed to predict the downstream effects of food additives on the human gut microbiome across 7 key targets:
1. `bifido_delta` (impact on *Bifidobacterium* abundance)
2. `lacto_delta` (impact on *Lactobacillus* abundance)
3. `akkermansia_delta` (impact on *Akkermansia muciniphila* abundance)
4. `enterobacteriaceae_delta` (impact on opportunistic *Enterobacteriaceae* abundance)
5. `diversity_delta` (impact on bacterial alpha diversity)
6. `scfa_delta` (impact on short-chain fatty acid production)
7. `barrier_risk` (impact on intestinal barrier permeability risk)

This report explains the data sources, the label generation process, the model training settings, and provides an analysis of the final evaluation metrics.

---
## 2. Dataset and Label Generation

The training pipeline relies on three main data sources:
1. **Additive Literature Coefficients**: `additives_effects.csv` compiles quantitative effects for $24$ target food additives compiled from major scientific publications (e.g., *Chassaing et al. 2015 Nature* for emulsifiers; *Suez et al. 2014 Nature* for artificial sweeteners).
2. **Product Additive Matrix**: `products_additives.csv` details the presence of food additives across $1,133$ products, represented as binary multi-hot feature vectors:
   $$X_{i, j} \in \{0, 1\} \quad \text{for product } i \text{ and additive } j$$
3. **Microbiome Labels Table**: Produced by `build_training_table.py` and stored in `products_microbiome_labels.csv`.

### The Linear-Additive Label Function
The label generation logic computes target values using a strictly linear, additive combination of individual additive effects:
$$Y_{i, t} = \sum_{j=1}^{24} X_{i, j} \cdot C_{j, t}$$
where $C_{j, t}$ is the constant effect coefficient of additive $j$ on target $t$ from `additives_effects.csv`. 

No stochastic noise, non-linear interactions, or dose-response curves are introduced during label generation.

---
## 3. Model Architectures & Training Settings

### 3.1. Multi-Output GBDT Regressor
Implemented in `train_microbiome_model.py`, this model consists of:
- **Wrapper**: `sklearn.multioutput.MultiOutputRegressor`
- **Base Estimator**: `sklearn.ensemble.GradientBoostingRegressor(max_depth=4, n_estimators=200, random_state=42)`
- **Train/Test Split**: 80% Train ($906$ products) / 20% Test ($227$ products) split randomly using seed $42$.

### 3.2. PyTorch Multilayer Perceptron (MLP)
Implemented in `train_microbiome_nn.py`, this model consists of:
- **Preprocessing**: Z-score normalization of features and targets based on training split statistics.
- **Architecture**:
  - Input Layer: $24$ units
  - Hidden Layer 1: $64$ units $\rightarrow$ ReLU Activation
  - Hidden Layer 2: $32$ units $\rightarrow$ ReLU Activation
  - Output Layer: $7$ units (fully connected)
- **Optimization**: AdamW optimizer, learning rate $= 2\times 10^{-3}$, weight decay $= 1\times 10^{-4}$, batch size $= 32$, trained for $600$ epochs.

---
## 4. Performance Results

The final evaluation metrics for the GBDT and PyTorch MLP models are stored in `train_stats.json` and `train_stats_nn.json`.

### 4.1. Gradient Boosting Regressor (GBDT) Results
- **Training Samples**: $906$
- **Testing Samples**: $227$

| Target Metric | $R^2$ Score | Mean Absolute Error (MAE) |
| :--- | :---: | :---: |
| `bifido_delta` | 0.99919 | 0.000625 |
| `lacto_delta` | 0.99959 | 0.000570 |
| `akkermansia_delta` | 0.99995 | 0.000109 |
| `enterobacteriaceae_delta` | 0.99948 | 0.000454 |
| `diversity_delta` | 0.99912 | 0.000454 |
| `scfa_delta` | 0.99950 | 0.000301 |
| `barrier_risk` | 0.99980 | 0.001080 |
| **Average** | **0.99952** | **0.000513** |

### 4.2. PyTorch MLP Results
- **Training Samples**: $906$
- **Testing Samples**: $227$

| Target Metric | $R^2$ Score | Mean Absolute Error (MAE) |
| :--- | :---: | :---: |
| `bifido_delta` | 0.99968 | 0.000213 |
| `lacto_delta` | 0.99981 | 0.000197 |
| `akkermansia_delta` | 0.99997 | 0.000045 |
| `enterobacteriaceae_delta` | 0.99983 | 0.000172 |
| `diversity_delta` | 0.99972 | 0.000169 |
| `scfa_delta` | 0.99982 | 0.000120 |
| `barrier_risk` | 0.99995 | 0.000379 |
| **Average** | **0.99983** | **0.000185** |

---
## 5. Explaining the Low MAE

While a test MAE near zero is uncommon in biological modeling, it is expected in this system because **the labels are generated synthetically using a strictly linear, noise-free combination of binary indicators**. 

Since the target $Y$ is a deterministic linear function of the binary features $X$, the mathematical mapping is:
$$Y = X \cdot C$$

Both the GBDT and PyTorch MLP models have more than sufficient capacity to solve this noiseless mapping:
* **The GBDT model** partitions the binary additive features, easily matching their linear combinations. The tiny residual error ($0.00051$ average MAE) is due to GBDT approximating a continuous output using decision trees.
* **The Neural Network** optimizes its weights to perfectly match the linear combination coefficients. The tiny remaining error ($0.00018$ average MAE) represents the minor optimization residuals from training for a finite number of epochs ($600$) with a small weight decay parameter ($1\times 10^{-4}$).

---
## 6. Conclusions
Both the multi-output Gradient Boosting Regressor and the PyTorch MLP models successfully learn the linear-additive relationship of food additives on gut microbiome targets. Achieving an average test MAE under $0.0006$ and $R^2 > 0.999$, these models are highly precise and ready for production deployment within the Gutsafe engine.

---
*Gutsafe Technical Report — June 2026*